In [1]:
import fsspec
fs = fsspec.filesystem("s3", anon=True)
oisst_files = fs.glob(
    "s3://noaa-cdr-sea-surface-temp-optimum-interpolation-pds/data/v2.1/avhrr/202503/oisst-avhrr-v02r01.*.nc"
)
oisst_files = sorted(["s3://" + f for f in oisst_files])

In [2]:
from dask_gateway import Gateway
gateway = Gateway()  # instantiate Dask gateway 
options = gateway.cluster_options()
cluster = gateway.new_cluster(options)
client = cluster.get_client()
cluster.adapt(minimum=4, maximum=30)
client = cluster.get_client()
client

Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /services/dask-gateway/clusters/prod.89d613d999344db9b7a7124dcd1dccba/status,


In [3]:
%%time
import dask
from virtualizarr import open_virtual_dataset
so = dict(anon=True, default_fill_cache=False, default_cache_type="none")
virtual_datasets = dask.compute(*[
    dask.delayed(open_virtual_dataset)(url, indexes={}, reader_options={"storage_options": so}, )
    for url in oisst_files
])

CPU times: user 378 ms, sys: 48.6 ms, total: 427 ms
Wall time: 1min 58s


In [4]:
# --- Cleanup ---
client.close()
cluster.shutdown()